# Basic processing pipeline for image-based profiles

The main objective of data processing is to get the data ready for downstream statistical analysis and machine learning. This means we want to nicely annotate the data with metadata of interest, to deal with any missing values, to try and adjust the data to remove technical variability (normalisation), and to get rid of features that add more noise than signal to our data. Data processing is usually a pipeline, where the output of each step serves as the input to the next step. 

Each step of the processing pipeline creates a different representation of the data, usually increasing data abstraction and refinement as we proceed. In this notebook, we will go from Level 3 to Level 5 data. The CellProfiler tutorial that you finished earlier took you from Level 1 to Level 2.




| **Data Description**                              | **Level** | **Details** |
| :------------------------------------------------ | :--------: | :----------- |
| **Images**                                        | Level 1 | Raw microscopy images captured from the Cell Painting assay. Each image corresponds to a single field of view in a specific well and channel. |
| **Single-cell profiles**                 | Level 2 | Per-cell morphological measurements extracted from images using for example *CellProfiler*. |
| **Aggregated profiles with metadata information** | Level 3 | Single-cell profiles aggregated to the **well level** (e.g., median across all cells in a well) and annotated with **metadata** such as plate, well, and treatment. |
| **Normalized aggregated profiles**                | Level 4a | Well-level profiles normalized relative to **control wells** (e.g., DMSO) to correct for plate or batch effects. |
| **Normalized and feature-selected profiles**      | Level 4b | Normalized profiles with redundant or noisy features removed to retain only the **most informative and stable features**. |
| **Consensus profiles**                            | Level 5 | Aggregation of replicate or multi-concentration profiles to create a **final consensus signature** per perturbation or treatment. Used for reproducibility assessment and biological interpretation. |

## Imports

In [ ]:
import pandas as pd
from pycytominer import normalize, feature_select, consensus

# Annotate well-level profiles (level 3)

In [ ]:
# Read in profiles from each plate
data_list = []
plates = ["BR00127145", "BR00127146", "BR00127147", "BR00127148", "BR00127149"]
for plate in plates:
    data_tmp = pd.read_parquet('../inputs/' + plate + '.parquet')
    data_list.append(data_tmp)
df_cellprofiler = pd.concat(data_list)
df_cellprofiler = df_cellprofiler.reset_index(drop = True)

In [ ]:
# Read in compound metadata
df_compound_info = pd.read_csv("../inputs/compound_info.csv")
df_compounds_wells_links = pd.read_csv('../inputs/compound_well_map.csv')
df_compounds_annotations = pd.read_csv("../inputs/perturbation_controls.csv")

In [ ]:
# merge df_cellprofiler and df_compounds_wells_links based on common columns Metadata_Source Metadata_Plate Metadata_Well
df_cellprofiler_compounds = pd.merge(
    df_cellprofiler,
    df_compounds_wells_links,
    on=["Metadata_Source", "Metadata_Plate", "Metadata_Well"],
    how="inner"
)

In [ ]:
# merge df_cellprofiler_compounds and df_compounds_wells_links based on common column Metadata_JCP2022
df_cellprofiler_compounds_infos = pd.merge(
    df_cellprofiler_compounds,
    df_compound_info,
    on=["Metadata_JCP2022"],
    how="inner"
)

In [ ]:
# merge df_cellprofiler_compounds_infos and df_compounds_annotations based on common column Metadata_JCP2022
df_level3 = pd.merge(
    df_cellprofiler_compounds_infos,
    df_compounds_annotations,
    on=["Metadata_JCP2022"],
    how="left"
)

# Save level 3 data to outputs folder as csv file
df_level3.to_csv('../outputs/df_level3.csv', index=False)

**Exercise** 
- How many unique compounds are included in this dataset? 
- How many replicates of each compound are there? 
- How are compound replicates arranged across the plates? Make a heatmap for each plate that shows how the compounds are plated. The heatmaps should have 384 cells, with rows and columns corresponding to the rows and columns of the physical plate.

## Normalize (level 4a)

To ensure consistent comparisons across plates and experiments, **feature values are normalized** relative to control wells — typically **DMSO** for chemical perturbations. Normalization attempts to remove plate-to-plate and systematic technical variability, allowing biological differences to be more accurately captured.

In [ ]:
# Normalize by plates
df_level4a_list = []
for plate in plates:
    df_level4a_plate = normalize(
        profiles=df_level3[df_level3["Metadata_Plate"] == plate],
        features="infer",
        meta_features="infer",
        samples="Metadata_Name == 'DMSO'",
        method="mad_robustize",
    )
    df_level4a_list.append(df_level4a_plate)

df_level4a = pd.concat(df_level4a_list).reset_index(drop=True)

In [ ]:
# Save level 4a data to outputs folder as csv file
df_level4a.to_csv('../outputs/df_level4a.csv', index=False)

**Exercise**
- Did normalisation change the amount of variability associated with plate in our dataset? Visualise just the negative control DMSO samples in two dimensions (ie. PCA or UMAP), before and after normalisation, with samples coloured by plate.

# Feature selection (level 4b)

CellProfiler extracts thousands of features, many of which can be **redundant or highly correlated**. To reduce noise and improve interpretability, **feature selection** is applied to retain only the most informative and non-redundant features. This step improves data quality and computational efficiency for downstream analysis.

In [26]:
feature_select_opts = [
    "variance_threshold",
    "drop_na_columns",
    "correlation_threshold",
    "blocklist",
    "drop_outliers",
]
df_level4b = feature_select(
    profiles=df_level4a, features="infer", samples="all", operation=feature_select_opts
)

In [ ]:
# Save level 4b data to outputs folder as csv file
df_level4b.to_csv('../outputs/df_level4b.csv', index=False)

**Exercise**
- Read [the documentation](https://pycytominer.readthedocs.io/en/stable/pycytominer.html#module-pycytominer.feature_select) to understand what each feature selection option is doing. 
- How many features does each feature selection option drop? Visualise which features each option drops with an upset plot.

# Consensus signature (level 5)

Finally, **well-level profiles** corresponding to the same **perturbation or treatment** (e.g., multiple wells, replicates, or concentrations) are **aggregated into consensus profiles**. These consensus profiles represent the **robust morphological signature** of each perturbation. Sometimes we prefer to use consensus profiles instead of replicate profiles for downstream analysis and visualisation.

In [ ]:
df_level5 = consensus(
    profiles=df_level4b,
    replicate_columns=["Metadata_JCP2022"],
    features="infer",
    operation="modz",
)

In [ ]:
# merge df_level5 and df_compounds_annotations based on common column Metadata_JCP2022
df_level5 = pd.merge(
    df_level5,
    df_compounds_annotations,
    on=["Metadata_JCP2022"],
    how="left"
)

# Save level 5 data to outputs folder as csv file
df_level5.to_csv('../outputs/df_level5.csv', index=False)